# 使用 Milvus 和 DeepSeek 构建 基于民法典的 RAG

### 准备数据

我们从 `mfd.md` 文件加载。

In [1]:
law_rules = {}

with open("mfd.md", 'r', encoding='utf-8') as file:
    chapter_id = ''
    group_id = ''
    rule_id = ''
    for line_num, line in enumerate(file, 1):
        clean_line = line.rstrip()
        clean_line = line.rstrip('\n')
        clean_line = line.lstrip()

        if clean_line.startswith('## ') or clean_line.startswith('### '):
            chapter_id = ''
            group_id = ''
            rule_id = ''
            continue
        elif clean_line.startswith('#### '):
            chapter_id = clean_line[5:].rstrip()
            group_id = ''
            rule_id = ''
            continue
        elif clean_line.startswith('##### '):
            group_id = clean_line[6:]
            rule_id = ''
            continue
        elif clean_line.startswith('**'):
            rule_id = clean_line[2:clean_line.rfind('**')]
            content = clean_line[clean_line.rfind('**') + 2 : ]
            law_rules[chapter_id + ' ' + group_id + ' ' + rule_id] = {
                "chapter_id": chapter_id,
                "group_id": group_id,
                "rule_id": rule_id,
                "content": content
            }
        elif len(clean_line) <= 0:
            continue
        else:
            if (len(chapter_id) > 0 and len(rule_id) > 0):
                key = chapter_id + ' ' + group_id + ' ' + rule_id
                law_rule = law_rules[key]
                law_rule['content'] = law_rule['content'] + clean_line
            
print(len(law_rules))

# 测试跨多行的条目是否正确
for item in law_rules.items():
    if item[1]['rule_id'] == '第五百五十条':
        print(item[1]['content'])

386
 债权人可以将合同的权利全部或者部分转让给第三人，但是有下列情形之一的除外：
（一）根据合同性质不得转让；
（二）按照当事人约定不得转让；
（三）依照法律规定不得转让。
债权人转让权利的，应当通知债务人。未经通知，该转让对债务人不发生效力。



### 准备 LLM 和 Embedding 模型

In [2]:
from openai import OpenAI
import os

deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1",  # DeepSeek API 的基地址
)

Milvus 自带的default embedding模型对中文不友好，改用deepseek的embedding模型。

In [3]:
from transformers import AutoModel, AutoTokenizer
import torch

MODEL_NAME = "thenlper/gte-small-zh"

# 加载Embedding模型
def load_embedding_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME)
    return tokenizer, model

# 生成文本嵌入向量
def get_embeddings(texts, tokenizer, model):
    inputs = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        return_tensors="pt",
        max_length=1024
    )
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # 使用[CLS]标记的向量作为整个文本的表示
    embeddings = outputs.last_hidden_state[:, 0, :]
    return embeddings.numpy()

tokenizer, model = load_embedding_model()


c:\Users\gigifrog\.conda\envs\dsstart\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
test_embedding = get_embeddings(["今天很热"], tokenizer, model)
embedding_dim = len(test_embedding[0])
print(embedding_dim)
print(test_embedding[0][:10])


512
[-0.02885137 -0.1524358  -0.63194406  0.04200799 -0.15200001  0.2230269
  0.01481954  0.05137496 -0.24334455 -0.16598058]


### 创建 Collection

In [5]:
from pymilvus import MilvusClient, FieldSchema, CollectionSchema, DataType, Collection, connections, utility

# milvus_client = MilvusClient(uri="http://127.0.0.1:19530")
connections.connect(host="127.0.0.1", port="19530")
collection_name = "my_mfd_rag_collection"

检查 collection 是否已存在，如果存在则删除它。

In [6]:
if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)

创建一个具有指定参数的新 collection。

In [7]:
fields = [
        # 主键字段 - 使用字符串类型作为主键
        FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, max_length=512),
        
        # 向量字段 - 假设使用512维浮点向量（根据实际需求修改）
        FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=embedding_dim),
        
        # 文本字段 - 存储中文字符
        FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=20000),

        FieldSchema(name="chapter_id", dtype=DataType.VARCHAR, max_length=100),
        FieldSchema(name="group_id", dtype=DataType.VARCHAR, max_length=100),
        FieldSchema(name="rule_id", dtype=DataType.VARCHAR, max_length=100)
    ]

schema = CollectionSchema(
        fields=fields,
        description="民法典数据库",
        enable_dynamic_field=True  # 允许动态字段
    )

try:
    collection = Collection(name=collection_name, schema=schema)
    print(f"✅ 成功创建集合: {collection_name}")
        
    index_params = {
        "index_type": "IVF_FLAT",
        "metric_type": "IP"
    }
        
    collection.create_index(field_name="vector", index_params=index_params)
    print("✅ 成功创建向量索引")
    
except Exception as e:
    print(f"❌ 创建集合失败: {e}")


✅ 成功创建集合: my_mfd_rag_collection
✅ 成功创建向量索引


### 插入数据

遍历文本行，创建嵌入，然后将数据插入 Milvus。

这里有一个新字段 `text`，它是在 collection schema 中未定义的字段。它将自动添加到保留的 JSON 动态字段中，该字段在高级别上可以被视为普通字段。

In [8]:
from tqdm import tqdm

data = []

for i, rule in enumerate(law_rules.items(), 1):
    data.append({"id": str(i), 
                 "vector": get_embeddings([rule[1]['content']], tokenizer, model)[0],
                 "text": rule[1]['content'],
                 "chapter_id": rule[1]['chapter_id'],
                 "group_id": rule[1]['group_id'],
                 "rule_id": rule[1]['rule_id']
                 })
collection.insert(data=data)

(insert count: 386, delete count: 0, upsert count: 0, timestamp: 459670912535363587, success count: 386, err count: 0

### 检索查询数据

In [9]:
question = "有哪些情形要约失效?"

在 collection 中搜索该问题，并检索语义上最匹配的前3个结果。

In [11]:
collection.load()
search_res = collection.search(
    data=get_embeddings([question], tokenizer, model),  # 将问题转换为嵌入向量
    anns_field="vector",
    param={"metric_type": "IP", "params": {}},
    limit= 5,  # 返回前3个结果
    output_fields=["text", "chapter_id", "group_id", "rule_id"],  # 返回 text 字段
)

让我们看一下查询的搜索结果

In [12]:
for res in search_res[0]:
    print(res['entity']['text'])


 有下列情形之一的，要约失效：
（一）要约人依法撤销要约；
（二）要约人拒绝要约；
（三）受要约人对要约的内容作出实质性变更；
（四）承诺期限届满，受要约人未作出承诺；
（五）受要约人拒绝要约。

 当事人对合同的效力可以约定附条件。
附生效条件的合同，自条件成就时生效。
附解除条件的合同，自条件成就时失效。
当事人为自己的利益不正当地阻止条件成就的，视为条件已经成就；不正当地促成条件成就的，视为条件不成就。

 合同中的下列免责条款无效：
（一）造成对方人身损害的；
（二）因故意或者重大过失造成对方财产损失的。

 要约到达受要约人时生效。
要约人发出要约后，可以撤回要约。撤回要约的通知应当在要约到达受要约人之前或者与要约同时到达受要约人。

 要约可以撤销，但是有下列情形之一的不得撤销：
（一）要约人确定承诺期限或者以其他形式明示要约不可撤销；
（二）受要约人有理由认为要约是不可撤销的，并已经为履行合同作了准备工作。



### 使用 LLM 获取 RAG 响应

将检索到的文档转换为字符串格式。

In [13]:
context = "\n".join(
    [res['entity']['text'] for res in search_res[0]]
)

In [14]:
context

' 有下列情形之一的，要约失效：\n（一）要约人依法撤销要约；\n（二）要约人拒绝要约；\n（三）受要约人对要约的内容作出实质性变更；\n（四）承诺期限届满，受要约人未作出承诺；\n（五）受要约人拒绝要约。\n\n 当事人对合同的效力可以约定附条件。\n附生效条件的合同，自条件成就时生效。\n附解除条件的合同，自条件成就时失效。\n当事人为自己的利益不正当地阻止条件成就的，视为条件已经成就；不正当地促成条件成就的，视为条件不成就。\n\n 合同中的下列免责条款无效：\n（一）造成对方人身损害的；\n（二）因故意或者重大过失造成对方财产损失的。\n\n 要约到达受要约人时生效。\n要约人发出要约后，可以撤回要约。撤回要约的通知应当在要约到达受要约人之前或者与要约同时到达受要约人。\n\n 要约可以撤销，但是有下列情形之一的不得撤销：\n（一）要约人确定承诺期限或者以其他形式明示要约不可撤销；\n（二）受要约人有理由认为要约是不可撤销的，并已经为履行合同作了准备工作。\n'

In [15]:
question

'有哪些情形要约失效?'

为语言模型定义系统和用户提示。此提示是使用从 Milvus 检索到的文档组装而成的。

In [16]:
SYSTEM_PROMPT = """
Human: 你是一个 AI 助手。你能够从提供的上下文段落片段中找到问题的答案。
"""
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。
<context>
{context}
</context>
<question>
{question}
</question>
"""

In [17]:
USER_PROMPT

'\n请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。\n<context>\n 有下列情形之一的，要约失效：\n（一）要约人依法撤销要约；\n（二）要约人拒绝要约；\n（三）受要约人对要约的内容作出实质性变更；\n（四）承诺期限届满，受要约人未作出承诺；\n（五）受要约人拒绝要约。\n\n 当事人对合同的效力可以约定附条件。\n附生效条件的合同，自条件成就时生效。\n附解除条件的合同，自条件成就时失效。\n当事人为自己的利益不正当地阻止条件成就的，视为条件已经成就；不正当地促成条件成就的，视为条件不成就。\n\n 合同中的下列免责条款无效：\n（一）造成对方人身损害的；\n（二）因故意或者重大过失造成对方财产损失的。\n\n 要约到达受要约人时生效。\n要约人发出要约后，可以撤回要约。撤回要约的通知应当在要约到达受要约人之前或者与要约同时到达受要约人。\n\n 要约可以撤销，但是有下列情形之一的不得撤销：\n（一）要约人确定承诺期限或者以其他形式明示要约不可撤销；\n（二）受要约人有理由认为要约是不可撤销的，并已经为履行合同作了准备工作。\n\n</context>\n<question>\n有哪些情形要约失效?\n</question>\n'

使用 DeepSeek 提供的 `deepseek-chat` 模型根据提示生成响应。

In [18]:
response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)

根据提供的上下文，要约失效的情形包括以下五种：

（一）要约人依法撤销要约；
（二）要约人拒绝要约；
（三）受要约人对要约的内容作出实质性变更；
（四）承诺期限届满，受要约人未作出承诺；
（五）受要约人拒绝要约。

（注：上下文第二点"要约人拒绝要约"与第五点"受要约人拒绝要约"重复，实际应为四种情形。根据《合同法》应为：1.要约被依法撤销；2.被拒绝；3.实质性变更；4.承诺期满未承诺）
